#  Canal YouTube — Automatización v3

**Flujo completo:**
```
Tema → Guion + keywords (Claude) → Voz (ElevenLabs) → Clips específicos (Pexels)
     → Música (Freesound) → Vídeo (MoviePy) → Miniatura → YouTube
```

### Instalación (ejecuta solo la primera vez)
```bash
pip install anthropic requests moviepy==1.0.3 imageio==2.25.1 imageio-ffmpeg==0.4.9 decorator==4.4.2 Pillow==9.5.0 google-api-python-client google-auth-httplib2 google-auth-oauthlib python-dotenv
```

###  Archivo .env necesario
```
ANTHROPIC_API_KEY=sk-ant-...
ELEVENLABS_API_KEY=...
PEXELS_API_KEY=...
FREESOUND_API_KEY=...
NASA_API_KEY=DEMO_KEY
```


IDEAS: 



##  CELDA 0 — Configuración y verificación de API Keys

In [ ]:
import os
import json
import requests
import random
import anthropic
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

ANTHROPIC_API_KEY  = os.getenv('ANTHROPIC_API_KEY')
ELEVENLABS_API_KEY = os.getenv('ELEVENLABS_API_KEY')
PEXELS_API_KEY     = os.getenv('PEXELS_API_KEY')
FREESOUND_API_KEY  = os.getenv('FREESOUND_API_KEY')
NASA_API_KEY       = os.getenv('NASA_API_KEY', 'DEMO_KEY')

WORK_DIR = Path('./videos')
WORK_DIR.mkdir(exist_ok=True)
(WORK_DIR / 'clips').mkdir(exist_ok=True)
(WORK_DIR / 'musica').mkdir(exist_ok=True)
(WORK_DIR / 'miniaturas').mkdir(exist_ok=True)

print('=== VERIFICACIÓN DE API KEYS ===')
for nombre, key in [
    ('Anthropic',  ANTHROPIC_API_KEY),
    ('ElevenLabs', ELEVENLABS_API_KEY),
    ('Pexels',     PEXELS_API_KEY),
    ('Freesound',  FREESOUND_API_KEY),
    ('NASA',       NASA_API_KEY),
]:
    estado = '✅' if key else '❌ FALTA'
    preview = key[:8] + '...' if key and len(key) > 8 else key
    print(f'  {estado} {nombre}: {preview}')
print(f'\n📁 Carpeta de trabajo: {WORK_DIR.resolve()}')

import os
from pathlib import Path
ffmpeg_bin = str(Path.home() / 'ffmpeg' / 'bin')
os.environ['PATH'] = ffmpeg_bin + os.pathsep + os.environ['PATH']
print(f'✅ ffmpeg con NVENC cargado desde: {ffmpeg_bin}')

---
##  CELDA 1 — Define el tema del vídeo


como puede existir la nada antes del bigbang
Que ocurre con el tiempo cuando alcanzas el horizonte de sucesos
Del vacio al hidrogeno, el violento nacimiento del universo desde el primer atomo- Documental


In [ ]:
# ============================================================
# ✏️  EDITA AQUÍ — Solo esta celda cambia en cada vídeo
# ============================================================

TEMA = ""

# ID de voz en ElevenLabs

VOICE_ID = "5egO01tkUjEzu7xSSE8M"  # Cambia por Voice ID preferido

# Duración objetivo del vídeo en minutos
DURACION_MIN = 12

# Clips por segmento del guion
CLIPS_POR_SEGMENTO = 3

# ============================================================
nombre_base = TEMA[:40].replace(' ', '_').replace('/', '-')
print(f'🎬 Tema: {TEMA}')
print(f'⏱️  Duración: {DURACION_MIN} min (~{DURACION_MIN * 150} palabras)')
print(f'🎞️  Clips totales estimados: {6 * CLIPS_POR_SEGMENTO}')
print(f'📁 Nombre base: {nombre_base}')

---
##  CELDA 2 — Generación del guion con Claude API
> Genera guion dividido en 6 segmentos con keywords visuales específicas para cada uno.

In [ ]:
def generar_guion(tema: str, duracion_min: int = 4,
                  modo_nocturno: bool = False) -> dict:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    if modo_nocturno:
        filosofia = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FILOSOFÍA DEL VÍDEO — MODO NOCTURNO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Este vídeo se escucha antes de dormir o de madrugada.
70% dato científico real con números concretos.
30% reflexión filosófica contemplativa y pausada.
Tono: íntimo, susurrado, como una conversación a las 3am.
No hay urgencia. No hay tensión agresiva. Solo profundidad.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ESTRUCTURA NOCTURNA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Segmento 0 — APERTURA SUAVE (máximo 3 frases)
  Empieza despacio. Sin golpe de efecto.
  Una observación tranquila que invite a reflexionar.
  Ejemplo: "Hay algo que ocurre cada noche mientras duermes..."

Segmento 1 — CURIOSIDAD 1
  Dato científico real con números concretos.
  Explicado con calma, sin prisa.
  Termina con una reflexión suave, no con una pregunta urgente.

Segmento 2 — CURIOSIDAD 2
  Dato más profundo que el anterior.
  Máximo 2 frases filosóficas contemplativas.
  El espectador debe sentir pequeñez y paz, no miedo.

Segmento 3 — CURIOSIDAD 3
  Dato que genera asombro tranquilo.
  La sensación debe ser de maravilla, no de vértigo.

Segmento 4 — CURIOSIDAD 4
  El dato más hermoso y perturbador a la vez.
  IMPORTANTE: Justo antes añade exactamente:
  "Suscríbete para no perderte lo que viene."
  Luego revela el dato. El CTA va ANTES de la revelación.

Segmento 5 — CIERRE CONTEMPLATIVO (40-60 segundos)
  No resuelve nada — abre una pregunta que acompaña al sueño.
  Termina con una frase que se quede en la cabeza al cerrar los ojos.
  Invita a comentar qué piensan mientras contemplan el universo.
"""
        tension = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TONO NOCTURNO — REGLAS ESTRICTAS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NUNCA uses ganchos agresivos ni urgencia.
NUNCA generes miedo o ansiedad.
SÍ usa asombro, contemplación y pequeñez cósmica.
SÍ usa pausas largas con puntos suspensivos.
El objetivo es hacer pensar suavemente, no impactar.
Frases más largas que en formato normal — 15-25 palabras.
Ritmo lento y pausado, como respirar despacio.
"""
    else:
        filosofia = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
FILOSOFÍA DEL VÍDEO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
70% dato científico real con números concretos.
30% reflexión filosófica puntual — máximo 2 frases por segmento.
Tono: documental científico con alma. No filosofía pura.
"""
        tension = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TENSIÓN NARRATIVA — LA REGLA MÁS IMPORTANTE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
El vídeo es una historia con tensión creciente, no una lista de datos.
Cada segmento debe terminar dejando algo sin resolver.
El espectador nunca debe sentir que tiene la respuesta completa
hasta el último segundo.

Flujo de tensión obligatorio:
- Segmento 0: dato impactante + pregunta sin responder
- Segmento 1: contexto que abre una pregunta mayor
- Segmento 2: responde parcialmente pero genera más intriga
- Segmento 3: la cosa se complica, la tensión aumenta
- Segmento 4: el dato más perturbador — máxima tensión
- Segmento 5: responde la pregunta central + deja algo abierto

NUNCA resuelvas la tensión antes del Segmento 5.
NUNCA des todos los datos en los primeros segmentos.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ESTRUCTURA DE CADA SEGMENTO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Segmento 0 — GANCHO (máximo 3 frases)
  Empieza con el dato más impactante. Termina con una pregunta
  que el espectador NECESITA responder.

Segmento 1 — CONTEXTO
  Explica dónde, qué y cuándo con números reales.
  Termina abriendo una pregunta más grande que el gancho.

Segmento 2 — PUNTO 1
  Dato científico concreto + máximo 2 frases filosóficas.
  Termina con una transición que aumenta la intriga.

Segmento 3 — PUNTO 2
  Dato más sorprendente que el anterior.
  La tensión se complica — no simplifica.

Segmento 4 — PUNTO 3
  El dato más perturbador del vídeo.
  El espectador debe sentir vértigo existencial aquí.
  IMPORTANTE: Justo antes del dato más impactante añade exactamente:
  "Suscríbete para no perderte lo que viene."
  Luego revela el dato. El CTA va ANTES de la revelación.

Segmento 5 — CIERRE (30-40 segundos)
  Responde la pregunta central.
  Termina con una idea que se quede en la cabeza días después.
  CTA natural al final — invita a comentar con una pregunta abierta.
"""

    # ← AQUÍ ESTÁ EL FIX: calculamos palabras objetivo
    palabras_min = duracion_min * 120
    palabras_max = duracion_min * 140
    palabras_por_segmento_min = (duracion_min * 120) // 6
    palabras_por_segmento_max = (duracion_min * 140) // 6

    prompt = f"""Eres un narrador documental inspirado en Carl Sagan.
Crea el guion completo para un vídeo de YouTube sobre: "{tema}"

{filosofia}
{tension}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DURACIÓN OBJETIVO — MUY IMPORTANTE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Duración objetivo: {duracion_min} minutos
- Palabras totales del guion: entre {palabras_min} y {palabras_max} palabras
- Ritmo de narración: documental pausado (120-140 palabras por minuto)
- Cada segmento debe tener entre {palabras_por_segmento_min} 
  y {palabras_por_segmento_max} palabras aproximadamente
- NO escribas guiones cortos. Si el guion tiene menos de {palabras_min} 
  palabras, NO has cumplido el objetivo.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
REGLAS DE ESCRITURA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Frases de 12-20 palabras con pausas naturales
- Usa puntos suspensivos (...) en momentos de máxima tensión
- Nunca uses listas, guiones ni asteriscos en el texto
- Números siempre en palabras:
    0.000000001% → "una millonésima parte de un por ciento"
    1.5 x 10^9   → "mil quinientos millones"
- Primera persona directa al espectador
- Nunca menciones: hook, segmento, estructura, drop-off
- NUNCA uses asteriscos, almohadillas, guiones bajos,
  corchetes ni ningún símbolo especial en el texto

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TÍTULO
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Nunca uses formato "TEMA: subtítulo"
- Genera incomodidad emocional o misterio
- Usa verbos de tensión: morirá, ya empezó, nadie sabe,
  es peor de lo que crees, ya está pasando
- Evita: misterios, secretos, increíble, asombroso
- Máximo 60 caracteres

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEYWORDS VISUALES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SIEMPRE temática espacial y cinematográfica.
NUNCA personas ni paisajes terrestres.
Añade siempre 'cinematic' o '4k' al final de cada keyword.
Nunca uses keywords genéricas de una sola palabra.

Formato correcto: '[objeto específico] [contexto] cinematic'

Ejemplos CORRECTOS:
  'dark cosmic void cinematic'
  'stellar nebula explosion 4k'
  'black hole accretion disk cinematic'
  'milky way galaxy timelapse cinematic'
  'dying star supernova collapse cinematic'
  'asteroid field deep space cinematic'
  'saturn rings closeup cinematic'

Ejemplos INCORRECTOS:
  'space', 'galaxy', 'nebula', 'universe' — demasiado genérico

Términos PROHIBIDOS: person, child, human, desert, ocean,
  tree, building, crowd, field, beach, forest, dark room, cave

Traducciones filosóficas cinematográficas:
  soledad        → 'lonely planet drifting deep space cinematic'
  insignificancia → 'pale blue dot earth cosmos cinematic'
  oscuridad      → 'infinite cosmic void darkness cinematic'
  tiempo         → 'galaxy formation timelapse cinematic'
  muerte/fin     → 'dying star supernova collapse cinematic'
  origen         → 'nebula star formation cinematic'
  escala         → 'galaxy cluster deep field hubble cinematic'

Para NASA — términos cortos 1-2 palabras:
   "sun", "nebula", "mars", "saturn", "spacecraft"
   "sun surface close up dramatic explosion"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Responde SOLO con JSON válido sin texto adicional:
{{
  "pregunta_central": "La pregunta que vertebra el vídeo",
  "titulo": "Título con gancho, máximo 60 caracteres",
  "titulo_miniatura": "Máximo 4 palabras MAYÚSCULAS",
  "titulo_alternativo": "Segundo título más clickbait",
  "descripcion": "Descripción SEO 150 palabras",
  "tags": ["tag1","tag2","tag3","tag4","tag5","tag6","tag7","tag8","tag9","tag10"],
  "primer_comentario": "Pregunta directa para comentarios",
  "keywords_nasa": ["keyword1", "keyword2"],
  "segmentos": [
    {{"id":0,"titulo":"Apertura", "texto":"...","keywords_pexels":["kw1","kw2","kw3","kw4"]}},
    {{"id":1,"titulo":"Curiosidad 1","texto":"...","keywords_pexels":["kw1","kw2","kw3","kw4"]}},
    {{"id":2,"titulo":"Curiosidad 2","texto":"...","keywords_pexels":["kw1","kw2","kw3","kw4"]}},
    {{"id":3,"titulo":"Curiosidad 3","texto":"...","keywords_pexels":["kw1","kw2","kw3","kw4"]}},
    {{"id":4,"titulo":"Curiosidad 4","texto":"...","keywords_pexels":["kw1","kw2","kw3","kw4"]}},
    {{"id":5,"titulo":"Cierre",   "texto":"...","keywords_pexels":["kw1","kw2","kw3","kw4"]}}
  ]
}}"""

    print(f' Generando guion para: "{tema}"...')
    print(f' Modo nocturno: {"ACTIVADO" if modo_nocturno else "desactivado"}')
    print(f' Objetivo: {palabras_min}-{palabras_max} palabras ({duracion_min} min)')

    # ← FIX: max_tokens dinámico según duración
    max_tokens = 16000 if duracion_min > 20 else 8000

    message = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = message.content[0].text.strip().replace('```json','').replace('```','').strip()
    datos = json.loads(raw)
    datos['guion'] = ' '.join([s['texto'] for s in datos['segmentos']])

    ruta = WORK_DIR / f"{nombre_base}_guion.json"
    with open(ruta, 'w', encoding='utf-8') as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)

    print(f' Guion generado ({len(datos["guion"].split())} palabras)')
    print(f' Pregunta central: {datos["pregunta_central"]}')
    print(f' Título: {datos["titulo"]}')
    print(f'  Miniatura: {datos["titulo_miniatura"]}')
    print(f' Keywords NASA: {datos["keywords_nasa"]}')
    print(f'\n Segmentos y keywords:')
    for s in datos['segmentos']:
        print(f'  [{s["id"]}] {s["titulo"]:12} → {s["keywords_pexels"]}')
    print(f'\n--- PREVIEW (primeras 200 palabras) ---')
    print(' '.join(datos['guion'].split()[:200]) + '...')
    return datos


datos_video = generar_guion(TEMA, DURACION_MIN)
GUION = datos_video['guion']

In [ ]:
import glob
chunks = sorted(glob.glob(str(WORK_DIR / "El_sonido*chunk*.mp3")))
print(chunks)

In [ ]:
import glob
mp3s = sorted(glob.glob(str(WORK_DIR / "El_sonido*.mp3")))
print(mp3s)

---
##  CELDA 3 — Generación de voz en off con ElevenLabs

In [ ]:
import re
import math

def limpiar_guion_tts(guion: str) -> str:
    guion = guion.replace('*', '')
    guion = guion.replace('#', '')
    guion = guion.replace('_', '')
    guion = re.sub(r'\[.*?\]', '', guion)
    guion = re.sub(r'<.*?>', '', guion)
    guion = re.sub(r' +', ' ', guion)
    guion = re.sub(r'\n +', '\n', guion)
    return guion.strip()

def dividir_en_chunks(texto: str, max_chars: int = 9000) -> list:
    if len(texto) <= max_chars:
        return [texto]
    
    chunks = []
    while texto:
        if len(texto) <= max_chars:
            chunks.append(texto.strip())
            break
        
        corte = texto[:max_chars].rfind('.')
        if corte == -1:
            corte = texto[:max_chars].rfind(',')
        if corte == -1:
            corte = texto[:max_chars].rfind(' ')
        
        chunks.append(texto[:corte+1].strip())
        texto = texto[corte+1:].strip()
    
    return chunks

def generar_voz(guion: str, voice_id: str, nombre_archivo: str) -> Path:
    url = f"https://api.elevenlabs.io/v1/text-to-speech/{voice_id}"
    headers = {
        "Accept": "audio/mpeg",
        "Content-Type": "application/json",
        "xi-api-key": ELEVENLABS_API_KEY
    }
    
    chunks = dividir_en_chunks(guion)
    print(f' Generando voz ({len(guion.split())} palabras)...')
    print(f' Dividido en {len(chunks)} chunks')
    print(' Espera...')

    audios = []
    for i, chunk in enumerate(chunks):
        print(f'  🔄 Chunk {i+1}/{len(chunks)} ({len(chunk)} caracteres)...')
        
        payload = {
            "text": chunk,
            "model_id": "eleven_multilingual_v2",
            "voice_settings": {
                "stability": 0.85,
                "similarity_boost": 0.75,
                "style": 0.05,
                "use_speaker_boost": True
            }
        }

        r = requests.post(url, json=payload, headers=headers)
        if r.status_code != 200:
            print(f'❌ Error ElevenLabs chunk {i+1}: {r.status_code}')
            print(r.text)
            raise Exception(f'Error ElevenLabs: {r.status_code}')

        ruta_chunk = WORK_DIR / f"{nombre_archivo}_chunk_{i}.mp3"
        with open(ruta_chunk, 'wb') as f:
            f.write(r.content)
        audios.append(ruta_chunk)
        print(f'   Chunk {i+1} generado')

    # Si solo hay un chunk, renombrar directamente
    if len(audios) == 1:
        ruta_final = WORK_DIR / f"{nombre_archivo}_voz.mp3"
        audios[0].rename(ruta_final)
        size_mb = ruta_final.stat().st_size / 1024 / 1024
        print(f' Audio generado: {ruta_final.name} ({size_mb:.1f} MB)')
        return ruta_final

    # Si hay varios chunks, unirlos con ffmpeg
    print(f' Uniendo {len(audios)} chunks...')
    lista_txt = WORK_DIR / f"{nombre_archivo}_chunks.txt"

    # ← FIX: rutas absolutas dentro del txt
    with open(lista_txt, 'w', encoding='utf-8') as f:
        for a in audios:
            f.write(f"file '{a.resolve()}'\n")

    ruta_final = WORK_DIR / f"{nombre_archivo}_voz.mp3"
    cmd = f'ffmpeg -y -f concat -safe 0 -i "{lista_txt.resolve()}" -c copy "{ruta_final.resolve()}"'
    print(f' Ejecutando ffmpeg...')
    resultado = os.system(cmd)
    print(f' Código de salida: {resultado}')

    # ← FIX: solo borrar chunks si el archivo final existe
    if ruta_final.exists():
        for a in audios:
            a.unlink()
        lista_txt.unlink()
        print(f' Chunks temporales eliminados')
    else:
        print(f' ffmpeg falló. Chunks conservados en:')
        for a in audios:
            print(f'   {a}')
        raise Exception(f'ffmpeg no generó el archivo final. Código: {resultado}')

    size_mb = ruta_final.stat().st_size / 1024 / 1024
    print(f' Audio final: {ruta_final.name} ({size_mb:.1f} MB)')
    return ruta_final


GUION_LIMPIO = limpiar_guion_tts(GUION)
print(f' Caracteres totales: {len(GUION_LIMPIO)}')
ruta_audio = generar_voz(GUION_LIMPIO, VOICE_ID, nombre_base)
print(f' Audio: {ruta_audio}')

In [ ]:
def buscar_nasa_videos(keyword: str, n: int = 3) -> list:
    """Busca vídeos en NASA — simplifica keywords para mejor resultado."""
    try:
        # Simplificar keyword a máximo 2 palabras para NASA
        keyword_simple = ' '.join(keyword.split()[:2])
        
        params = {"q": keyword_simple, "media_type": "video", "page_size": 20}
        r = requests.get("https://images-api.nasa.gov/search",
                        params=params, timeout=10)
        if r.status_code != 200:
            return []

        items = r.json().get('collection', {}).get('items', [])
        random.shuffle(items)
        urls = []

        for item in items[:n*3]:
            col_url = item.get('href')
            if not col_url:
                continue
            try:
                col_r = requests.get(col_url, timeout=8)
                if col_r.status_code != 200:
                    continue
                archivos = col_r.json()
                mp4s = [f for f in archivos if f.endswith('.mp4')
                        and '~' not in f]
                if mp4s:
                    mobile = [f for f in mp4s if 'mobile' in f.lower()
                              or 'small' in f.lower() or '480' in f]
                    url = mobile[0] if mobile else mp4s[0]
                    urls.append(url)
                    if len(urls) >= n:
                        break
            except:
                continue
        return urls
    except:
        return []


def buscar_nasa_svs(keyword: str) -> list:
    """NASA Scientific Visualization Studio — simplifica keywords."""
    try:
        # Simplificar keyword a máximo 2 palabras para SVS
        keyword_simple = ' '.join(keyword.split()[:2])
        
        url = "https://svs.gsfc.nasa.gov/api/v1/media/"
        params = {"search": keyword_simple, "format": "json"}
        r = requests.get(url, params=params, timeout=10)
        if r.status_code != 200:
            return []

        items = r.json().get('results', [])
        random.shuffle(items)
        urls = []

        for item in items[:5]:
            archivos = item.get('media_files', [])
            mp4s = [f['url'] for f in archivos
                   if f.get('url', '').endswith('.mp4')]
            if mp4s:
                urls.append(mp4s[0])
        return urls
    except:
        return []


def buscar_pexels_espacio(keyword: str, n: int = 2) -> list:
    """
    Busca clips en Pexels.
    Solo añade 'space' si la keyword es claramente espacial.
    """
    headers = {"Authorization": PEXELS_API_KEY}

    # Solo forzar 'space' si la keyword ya es espacial
    palabras_espacio = ['star', 'planet', 'galaxy', 'nebula', 'cosmos',
                        'solar', 'sun', 'moon', 'asteroid', 'comet',
                        'supernova', 'black hole', 'spacecraft', 'nasa',
                        'universe', 'orbit', 'telescope', 'void', 'cosmic']

    es_espacial = any(p in keyword.lower() for p in palabras_espacio)
    query = f"{keyword} space" if es_espacial else keyword

    params = {
        "query": query,
        "per_page": n,
        "page": random.randint(1, 5),
        "orientation": "landscape",
        "size": "medium"
    }
    try:
        r = requests.get("https://api.pexels.com/videos/search",
                        headers=headers, params=params, timeout=10)
        if r.status_code != 200:
            return []
        videos = r.json().get('videos', [])
        urls = []
        for v in videos:
            files = [f for f in v.get('video_files', [])
                     if f.get('height', 0) <= 1080
                     and f.get('file_type') == 'video/mp4']
            files.sort(key=lambda x: x.get('height', 0), reverse=True)
            if files:
                urls.append((v['id'], files[0]['link']))
        return urls
    except:
        return []


def buscar_clips_pixabay(keyword: str, n: int = 2) -> list:
    """Busca clips en Pixabay Videos — gratis y sin copyright."""
    try:
        # Solo añadir 'space' si la keyword es espacial
        palabras_espacio = ['star', 'planet', 'galaxy', 'nebula', 'solar',
                           'sun', 'moon', 'asteroid', 'supernova', 'cosmic']
        es_espacial = any(p in keyword.lower() for p in palabras_espacio)
        query = f"{keyword} space" if es_espacial else keyword

        params = {
            "key": PIXABAY_API_KEY,
            "q": query,
            "video_type": "film",
            "per_page": n,
            "safesearch": "true"
        }
        r = requests.get("https://pixabay.com/api/videos/",
                        params=params, timeout=10)
        if r.status_code != 200:
            return []

        hits = r.json().get('hits', [])
        urls = []
        for v in hits:
            videos = v.get('videos', {})
            url = (videos.get('large', {}).get('url') or
                   videos.get('medium', {}).get('url'))
            if url:
                urls.append((v['id'], url))
        return urls
    except:
        return []


def descargar_video_nasa(url: str, carpeta: Path, seg_id: int, idx: int) -> str | None:
    """Descarga un vídeo de la NASA."""
    nombre = url.split('/')[-1].split('?')[0][:50]
    ruta = carpeta / f"nasa_seg{seg_id:02d}_{idx:02d}_{nombre}"
    if ruta.exists() and ruta.stat().st_size > 10000:
        return str(ruta)
    try:
        r = requests.get(url, stream=True, timeout=60)
        if r.status_code == 200:
            with open(ruta, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536):
                    f.write(chunk)
            if ruta.stat().st_size > 10000:
                return str(ruta)
            else:
                ruta.unlink()
    except:
        pass
    return None


def descargar_clip_pexels(video_id: int, url: str, carpeta: Path,
                           seg_id: int, idx: int) -> str | None:
    """Descarga un clip de Pexels o Pixabay."""
    ruta = carpeta / f"pex_seg{seg_id:02d}_{idx:02d}_{video_id}.mp4"
    if ruta.exists():
        return str(ruta)
    try:
        r = requests.get(url, stream=True, timeout=30)
        if r.status_code == 200:
            with open(ruta, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            return str(ruta)
    except:
        pass
    return None


def descargar_clips_por_segmento(segmentos: list, clips_por_seg: int = 3) -> list:
    """
    Para cada segmento busca clips en este orden de prioridad:
      1. NASA Image Library (keywords simplificadas)
      2. NASA SVS (keywords simplificadas)
      3. Pixabay Videos
      4. Pexels (sin forzar space en keywords no espaciales)
      5. Fallback "sun" o "nebula"
    """
    import shutil

    carpeta = WORK_DIR / 'clips'

    # Limpiar caché para evitar reutilización
    if carpeta.exists():
        shutil.rmtree(carpeta)
        print('🗑️  Caché de clips limpiada')
    carpeta.mkdir(exist_ok=True)

    clips_ordenados = []

    print(f'🎬 Descargando clips para {len(segmentos)} segmentos...')
    print('   Fuentes: NASA Library → NASA SVS → Pixabay → Pexels')
    print()

    for seg in segmentos:
        seg_id   = seg['id']
        titulo   = seg['titulo']
        keywords = seg['keywords_pexels']
        clips_seg = []

        print(f'  📍 [{seg_id}] {titulo}')

        # ── 1. NASA Image Library ────────────────────────────
        for kw in keywords[:2]:
            if len(clips_seg) >= clips_por_seg * 2:
                break
            nasa_urls = buscar_nasa_videos(kw, n=clips_por_seg)
            if nasa_urls:
                print(f'     🌌 NASA Library "{kw}" → {len(nasa_urls)} clips')
                for i, url in enumerate(nasa_urls[:clips_por_seg]):
                    ruta = descargar_video_nasa(url, carpeta, seg_id,
                                               len(clips_seg))
                    if ruta:
                        clips_seg.append((seg_id, ruta))
            else:
                print(f'     ⚠️  NASA Library sin resultados para "{kw}"')

        # ── 2. NASA SVS ──────────────────────────────────────
        if len(clips_seg) < clips_por_seg:
            print(f'     🔭 Buscando en NASA SVS...')
            for kw in keywords[:2]:
                if len(clips_seg) >= clips_por_seg * 2:
                    break
                svs_urls = buscar_nasa_svs(kw)
                if svs_urls:
                    print(f'     🌌 NASA SVS "{kw}" → {len(svs_urls)} clips')
                    for url in svs_urls[:2]:
                        ruta = descargar_video_nasa(url, carpeta, seg_id,
                                                   len(clips_seg) + 50)
                        if ruta:
                            clips_seg.append((seg_id, ruta))
                else:
                    print(f'     ⚠️  NASA SVS sin resultados para "{kw}"')

        # ── 3. Pixabay ───────────────────────────────────────
        if len(clips_seg) < clips_por_seg:
            print(f'     🎥 Buscando en Pixabay...')
            for kw in keywords:
                if len(clips_seg) >= clips_por_seg * 2:
                    break
                pixabay_results = buscar_clips_pixabay(kw, n=2)
                if pixabay_results:
                    print(f'     📦 Pixabay "{kw}" → {len(pixabay_results)} clips')
                    for vid_id, url in pixabay_results:
                        ruta = descargar_clip_pexels(vid_id, url, carpeta,
                                                    seg_id, len(clips_seg) + 200)
                        if ruta:
                            clips_seg.append((seg_id, ruta))

        # ── 4. Pexels ────────────────────────────────────────
        if len(clips_seg) < clips_por_seg:
            print(f'     🎬 Complementando con Pexels...')
            for kw in keywords:
                if len(clips_seg) >= clips_por_seg * 3:
                    break
                resultados = buscar_pexels_espacio(kw, n=2)
                for vid_id, url in resultados:
                    ruta = descargar_clip_pexels(vid_id, url, carpeta,
                                                seg_id, len(clips_seg) + 300)
                    if ruta:
                        clips_seg.append((seg_id, ruta))
                if resultados:
                    print(f'        Pexels "{kw}" → {len(resultados)} clips')

        # ── 5. Fallback final ────────────────────────────────
        if not clips_seg:
            print(f'     🔄 Fallback espacial...')
            for kw_fallback in ['sun', 'nebula', 'galaxy', 'earth orbit']:
                for url in buscar_nasa_videos(kw_fallback, n=2):
                    ruta = descargar_video_nasa(url, carpeta, seg_id, 99)
                    if ruta:
                        clips_seg.append((seg_id, ruta))
                if clips_seg:
                    break

        clips_ordenados.extend(clips_seg)
        print(f'     ✅ {len(clips_seg)} clips para este segmento')
        print()

    clips_ordenados.sort(key=lambda x: x[0])
    lista = [r for (_, r) in clips_ordenados]
    print(f'✅ Total clips: {len(lista)}')
    print(f'   Fuentes: NASA Library + NASA SVS + Pixabay + Pexels')
    return lista


lista_clips = descargar_clips_por_segmento(datos_video['segmentos'], CLIPS_POR_SEGMENTO)
print(f'\n📦 Clips listos: {len(lista_clips)}')

---
## 🎬 CELDA 5 — Montaje del vídeo + Música épica (Freesound)
> Música épica cinematográfica diferente en cada vídeo. Fallback automático si falla.

In [ ]:
from moviepy.editor import AudioFileClip
import subprocess
import random

VOLUMEN_MUSICA = 0.18
CARPETA_MUSICA_LOCAL = Path(r"")


def descargar_musica(carpeta: Path) -> Path | None:
    tracks = list(CARPETA_MUSICA_LOCAL.glob('*.mp3'))
    if not tracks:
        print('  ⚠️  No se encontraron MP3 en la carpeta')
        return None
    track = random.choice(tracks)
    print(f'  🎵 Música seleccionada: {track.name}')
    return track


def montar_video(clips_paths: list, audio_path: Path, nombre_salida: str) -> Path:
    print('🎬 Iniciando montaje...')

    # Duración total
    tmp = AudioFileClip(str(audio_path))
    duracion_total = tmp.duration
    tmp.close()
    print(f'  ⏱️  Duración: {duracion_total:.1f}s ({duracion_total/60:.1f} min)')

    # Música
    print('  🎵 Preparando música...')
    ruta_musica = descargar_musica(WORK_DIR)

    # Calcular cuántos clips necesitamos
    n        = len(clips_paths)
    dur_clip = max(3, min(6, duracion_total / n))
    needed   = int(duracion_total / dur_clip) + 2
    extended = (clips_paths * (needed // n + 1))[:needed]
    print(f'  🎞️  {n} clips disponibles × {dur_clip:.1f}s → {len(extended)} clips a usar')

    # Procesar cada clip con ffmpeg individualmente (sin cargar en RAM)
    print('  📹 Procesando clips con ffmpeg...')
    clips_temp = []
    t_actual = 0

    for i, cp in enumerate(extended):
        if t_actual >= duracion_total:
            break
        dur = min(dur_clip, duracion_total - t_actual)
        ruta_clip_temp = WORK_DIR / f"_clip_temp_{i:03d}.mp4"

        cmd_clip = [
            'ffmpeg', '-y',
            '-i', cp,
            '-t', str(dur),
            '-vf', 'scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080',
            '-r', '30',
            '-an',
            '-c:v', 'libx264',
            '-preset', 'ultrafast',
            '-crf', '28',
            str(ruta_clip_temp)
        ]

        result = subprocess.run(cmd_clip, capture_output=True, text=True)
        if result.returncode == 0 and ruta_clip_temp.exists():
            clips_temp.append(ruta_clip_temp)
            t_actual += dur
        else:
            print(f'    ⚠️  Clip {i} falló, saltando...')

    if not clips_temp:
        raise Exception('❌ Sin clips procesados')

    print(f'  ✅ {len(clips_temp)} clips procesados')

    # Concatenar todos los clips con ffmpeg concat
    print('  🔗 Concatenando clips...')
    lista_txt = WORK_DIR / f"{nombre_salida}_lista_clips.txt"
    with open(lista_txt, 'w', encoding='utf-8') as f:
        for c in clips_temp:
            f.write(f"file '{c.resolve()}'\n")

    ruta_temp = WORK_DIR / f"{nombre_salida}_temp_raw.mp4"
    cmd_concat = [
        'ffmpeg', '-y',
        '-f', 'concat',
        '-safe', '0',
        '-i', str(lista_txt.resolve()),
        '-t', str(duracion_total),
        '-c', 'copy',
        str(ruta_temp)
    ]
    result = subprocess.run(cmd_concat, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  ❌ Error concatenando: {result.stderr[-300:]}')
        raise Exception('Error en concatenación')

    # Limpiar clips temporales
    for c in clips_temp:
        try:
            c.unlink()
        except:
            pass
    lista_txt.unlink()
    print('  🧹 Clips temporales eliminados')

    # Mezclar vídeo + voz + música
    print('  🎚️  Mezclando audio con ffmpeg NVENC...')
    ruta_salida = WORK_DIR / f"{nombre_salida}_final.mp4"

    if ruta_musica and Path(str(ruta_musica)).exists():
        cmd = [
            'ffmpeg', '-y',
            '-i', str(ruta_temp),
            '-i', str(audio_path),
            '-stream_loop', '-1',
            '-i', str(ruta_musica),
            '-filter_complex',
            f'[1:a]volume=1.0[voz];'
            f'[2:a]volume={VOLUMEN_MUSICA},afade=t=out:st={duracion_total-4}:d=4[musica];'
            f'[voz][musica]amix=inputs=2:duration=first[audio_final]',
            '-map', '0:v',
            '-map', '[audio_final]',
            '-c:v', 'h264_nvenc',
            '-preset', 'p4',
            '-rc', 'vbr',
            '-cq', '23',
            '-b:v', '0',
            '-c:a', 'aac',
            '-b:a', '192k',
            '-shortest',
            str(ruta_salida)
        ]
        print(f'  🎵 Voz 100% + música {int(VOLUMEN_MUSICA*100)}% (loop activado)')
    else:
        cmd = [
            'ffmpeg', '-y',
            '-i', str(ruta_temp),
            '-i', str(audio_path),
            '-map', '0:v',
            '-map', '1:a',
            '-c:v', 'h264_nvenc',
            '-preset', 'p4',
            '-rc', 'vbr',
            '-cq', '23',
            '-b:v', '0',
            '-c:a', 'aac',
            '-b:a', '192k',
            '-shortest',
            str(ruta_salida)
        ]
        print('  🔇 Solo voz')

    print('  🚀 Codificando con GPU (NVENC)...')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'  ⚠️  NVENC falló — usando libx264...')
        print(f'  Error: {result.stderr[-200:]}')

        if ruta_musica and Path(str(ruta_musica)).exists():
            cmd_fallback = [
                'ffmpeg', '-y',
                '-i', str(ruta_temp),
                '-i', str(audio_path),
                '-stream_loop', '-1',
                '-i', str(ruta_musica),
                '-filter_complex',
                f'[1:a]volume=1.0[voz];'
                f'[2:a]volume={VOLUMEN_MUSICA},afade=t=out:st={duracion_total-4}:d=4[musica];'
                f'[voz][musica]amix=inputs=2:duration=first[audio_final]',
                '-map', '0:v',
                '-map', '[audio_final]',
                '-c:v', 'libx264',
                '-preset', 'fast',
                '-crf', '23',
                '-c:a', 'aac',
                '-b:a', '192k',
                '-shortest',
                str(ruta_salida)
            ]
        else:
            cmd_fallback = [
                'ffmpeg', '-y',
                '-i', str(ruta_temp),
                '-i', str(audio_path),
                '-map', '0:v',
                '-map', '1:a',
                '-c:v', 'libx264',
                '-preset', 'fast',
                '-crf', '23',
                '-c:a', 'aac',
                '-b:a', '192k',
                '-shortest',
                str(ruta_salida)
            ]

        result2 = subprocess.run(cmd_fallback, capture_output=True, text=True)
        if result2.returncode != 0:
            print(f'  ❌ libx264 también falló:')
            print(result2.stderr[-300:])
            raise Exception('Fallo en codificación final')

    if ruta_temp.exists():
        ruta_temp.unlink()

    size_mb = ruta_salida.stat().st_size / 1024 / 1024
    print(f'\n✅ Vídeo listo: {ruta_salida.name} ({size_mb:.0f} MB)')
    print(f'🎵 Música al {int(VOLUMEN_MUSICA*100)}% de volumen')
    return ruta_salida


ruta_video_final = montar_video(lista_clips, ruta_audio, nombre_base)


---
##  CELDA 6 — Metadatos para YouTube Studio

In [ ]:
def mostrar_metadatos(datos: dict):
    sep = '=' * 60
    print(sep)
    print('📺 METADATOS PARA YOUTUBE STUDIO')
    print(sep)
    print(f'\n  TÍTULO PRINCIPAL:')
    print(f'   {datos["titulo"]}')
    print(f'\n  TÍTULO ALTERNATIVO:')
    print(f'   {datos["titulo_alternativo"]}')
    print(f'\n DESCRIPCIÓN:')
    print('-' * 40)
    print(datos['descripcion'])
    print('\n Suscríbete para más curiosidades del cosmos.')
    print('\n LINKS DE AFILIADO:')
    print('   • Telescopio recomendado: [TU LINK AMAZON]')
    print('   • Libro de astronomía: [TU LINK AMAZON]')
    print('-' * 40)
    print(f'\n TAGS:')
    print(', '.join(datos['tags']))
    print(f'\n COMENTARIO FIJADO:')
    print(f'   {datos["primer_comentario"]}')
    print(f'\n{sep}')
    print('  CONFIGURACIÓN YOUTUBE STUDIO:')
    print('   • Categoría: Ciencia y tecnología')
    print('   • Idioma: Español')
    print('   • Público: Para todos los públicos')
    print('   • Mejor hora: Martes/Viernes 18:00-20:00 España')
    print(sep)

    ruta = WORK_DIR / f"{nombre_base}_metadatos.txt"
    with open(ruta, 'w', encoding='utf-8') as f:
        f.write(f"TÍTULO: {datos['titulo']}\n\n")
        f.write(f"TÍTULO ALT: {datos['titulo_alternativo']}\n\n")
        f.write(f"DESCRIPCIÓN:\n{datos['descripcion']}\n\n")
        f.write(f"TAGS: {', '.join(datos['tags'])}\n\n")
        f.write(f"COMENTARIO: {datos['primer_comentario']}\n")
    print(f'\n Metadatos guardados: {ruta.name}')


mostrar_metadatos(datos_video)

---
##  CELDA 7 — Subida automática a YouTube (opcional)
> Requiere configurar OAuth. Ver instrucciones en la guía Word.

In [ ]:
import pickle
from datetime import datetime, timedelta
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

SCOPES = ['https://www.googleapis.com/auth/youtube.upload']


def autenticar_youtube():
    creds = None
    if os.path.exists('youtube_token.pickle'):
        with open('youtube_token.pickle', 'rb') as f:
            creds = pickle.load(f)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'client_secrets.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('youtube_token.pickle', 'wb') as f:
            pickle.dump(creds, f)
    return build('youtube', 'v3', credentials=creds)


def subir_youtube(youtube, video_path, datos, horas=24):
    fecha = datetime.utcnow() + timedelta(hours=horas)
    body = {
        'snippet': {
            'title': datos['titulo'],
            'description': datos['descripcion'],
            'tags': datos['tags'],
            'categoryId': '28',
            'defaultLanguage': 'es'
        },
        'status': {
            'privacyStatus': 'private',
            'publishAt': fecha.strftime('%Y-%m-%dT%H:%M:%S.000Z'),
            'selfDeclaredMadeForKids': False
        }
    }

    media = MediaFileUpload(
        str(video_path),
        mimetype='video/mp4',
        resumable=True
    )

    print(f'  Subiendo vídeo a YouTube...')
    print(f' Se publicará en {horas} horas')
    print(' Esto puede tardar varios minutos...')

    request = youtube.videos().insert(
        part='snippet,status',
        body=body,
        media_body=media
    )

    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f'  📤 Progreso: {int(status.progress()*100)}%')

    video_id = response['id']
    print(f'\n Vídeo subido correctamente')
    print(f' https://www.youtube.com/watch?v={video_id}')
    print(f' Se publicará automáticamente en {horas} horas')
    return video_id


# ── EJECUTAR ──────────────────────────────────────────────
yt = autenticar_youtube()
subir_youtube(yt, ruta_video_final, datos_video, horas=24)

---
##  CELDA 8 — Resumen final

In [ ]:
print('=' * 60)
print('🚀 VÍDEO COMPLETADO')
print('=' * 60)
print(f'\n📌 Tema: {TEMA}')
print(f'📺 Título: {datos_video["titulo"]}')
print(f'\n📁 Archivos generados:')
archivos = [
    (ruta_audio,           '🔊 Voz en off'),
    (ruta_video_final,     '🎬 Vídeo final'),
    (WORK_DIR / f'{nombre_base}_guion.json',    '📄 Guion'),
    (WORK_DIR / f'{nombre_base}_metadatos.txt', '📋 Metadatos'),
]
for ruta, label in archivos:
    p = Path(str(ruta))
    if p.exists():
        size = p.stat().st_size / 1024 / 1024
        print(f'   {label}: {p.name} ({size:.1f} MB)')
print(f'\n📦 Clips: {len(lista_clips)} (en videos/clips/)')
print('\n📋 PRÓXIMOS PASOS:')
print('   2. 📤  Sube el vídeo a YouTube Studio')
print('   3. 🖼️  Sube la miniatura')
print('   4. 📅  Programa: martes/viernes 18:00-20:00 España')
print('   5. 💬  Publica el comentario fijado')
print('   6. 🔄  Cambia TEMA en Celda 1 para el siguiente vídeo')
print('\n' + '=' * 60)
print('=' * 60)

In [ ]:
# ============================================================
#  CELDA LIMPIEZA — Borra archivos del vídeo ya subido
# ============================================================
# Ejecutar SOLO después de confirmar que el vídeo está
# correctamente subido y programado en YouTube Studio
# ============================================================

import shutil

def limpiar_carpeta_video(work_dir: Path, confirmar: bool = False):
    if not confirmar:
        print('  Para confirmar la limpieza cambia confirmar=True')
        print(f' Carpeta a limpiar: {work_dir.resolve()}')
        print('\n Archivos que se borrarán:')
        total_mb = 0
        for f in work_dir.rglob('*'):
            if f.is_file():
                size_mb = f.stat().st_size / 1024 / 1024
                total_mb += size_mb
                print(f'   • {f.name} ({size_mb:.1f} MB)')
        print(f'\n💾 Total: {total_mb:.0f} MB')
        return

    # Carpetas a limpiar — NO borra el .env ni los secrets
    carpetas = ['clips', 'audio', 'musica']
    archivos_ext = ['.mp4', '.mp3', '.m4a', '.json']

    print(' Limpiando archivos del vídeo...')
    total_borrado = 0

    # Borrar carpetas de clips y audio
    for carpeta in carpetas:
        ruta = work_dir / carpeta
        if ruta.exists():
            size = sum(
                f.stat().st_size for f in ruta.rglob('*') if f.is_file()
            ) / 1024 / 1024
            shutil.rmtree(ruta)
            ruta.mkdir(exist_ok=True)
            total_borrado += size
            print(f'  ✅ Carpeta limpiada: {carpeta}/ ({size:.0f} MB)')

    # Borrar archivos sueltos en WORK_DIR
    for f in work_dir.iterdir():
        if f.is_file() and f.suffix in archivos_ext:
            size = f.stat().st_size / 1024 / 1024
            f.unlink()
            total_borrado += size
            print(f'  ✅ Borrado: {f.name} ({size:.0f} MB)')

    print(f'\n Limpieza completada — {total_borrado:.0f} MB liberados')
    print(f' Carpeta output conservada (vídeo final)')


# ── EJECUTAR ──────────────────────────────────────────────
# Paso 1: ejecuta con confirmar=False para ver qué se borrará
# Paso 2: cambia a confirmar=True para borrar

limpiar_carpeta_video(WORK_DIR, confirmar=False)

In [ ]:
# ============================================================
#  CELDA 9 — Shorts desde el vídeo largo
# ============================================================

import librosa
import torch
import gc
import subprocess
from moviepy.editor import VideoFileClip, AudioFileClip

N_SHORTS_VIDEO     = 6    # Shorts a generar por vídeo
DURACION_SHORT_V   = 55   # segundos
HORA_SHORTS        = 13   # hora de publicación
DIAS_ENTRE_SHORTS  = 1    # días entre cada Short


def detectar_momentos(audio_path: Path, n: int = 3) -> list:
    print(f'🎵 Detectando momentos impactantes...')
    y, sr      = librosa.load(str(audio_path), sr=None)
    frame_len  = int(sr * 2)
    hop_len    = int(sr * 0.5)
    rms        = librosa.feature.rms(y=y, frame_length=frame_len,
                                      hop_length=hop_len)[0]
    tiempos    = librosa.frames_to_time(range(len(rms)), sr=sr,
                                         hop_length=hop_len)
    duracion   = librosa.get_duration(y=y, sr=sr)
    margen     = DURACION_SHORT_V + 5

    validos = [(t, e) for t, e in zip(tiempos, rms)
               if t + margen < duracion and t > 5]
    if not validos:
        return [10.0]

    validos.sort(key=lambda x: x[1], reverse=True)
    seleccionados = []
    for t, e in validos:
        if not any(abs(t - s) < 60 for s in seleccionados):
            seleccionados.append(t)
        if len(seleccionados) >= n:
            break

    seleccionados.sort()
    print(f'  ✅ Momentos: {[f"{t:.1f}s" for t in seleccionados]}')
    return seleccionados


def crear_short_desde_video(video_path: Path, audio_path: Path,
                             inicio: float, duracion: float,
                             indice: int, datos: dict) -> Path:
    from PIL import Image, ImageDraw, ImageFont
    import numpy as np

    print(f'\n  🎬 Generando Short {indice+1} desde vídeo largo...')

    video = VideoFileClip(str(video_path)).subclip(inicio, inicio + duracion)

    ANCHO      = 1080
    ALTO       = 1920
    ancho_orig = video.w
    alto_orig  = video.h

    titulo_pantalla = datos.get('titulo_miniatura', datos.get('titulo', ''))[:60]

    color_badge = [(20,60,180),(80,20,160),(20,100,120),(140,20,60),(20,80,40)]
    badge_color = color_badge[indice % len(color_badge)]

    def procesar_frame(frame, t):
        from PIL import Image, ImageDraw, ImageFont
        img = Image.new('RGB', (ANCHO, ALTO), (0,0,0))

        ratio = max(ANCHO / ancho_orig, (ALTO * 0.40) / alto_orig)
        nw    = int(ancho_orig * ratio)
        nh    = int(alto_orig  * ratio)
        img_v = Image.fromarray(frame).resize((nw, nh), Image.LANCZOS)

        y_vid = (ALTO - nh) // 2
        x_off = (ANCHO - nw) // 2
        if nw > ANCHO:
            x_crop = (nw - ANCHO) // 2
            img_v  = img_v.crop((x_crop, 0, x_crop + ANCHO, nh))
            x_off  = 0
        img.paste(img_v, (x_off, y_vid))

        draw = ImageDraw.Draw(img)
        try:
            fb = ImageFont.truetype("C:/Windows/Fonts/impact.ttf", 32)
            ft = ImageFont.truetype("C:/Windows/Fonts/impact.ttf", 68)
            fs = ImageFont.truetype("C:/Windows/Fonts/impact.ttf", 42)
        except:
            fb = ft = fs = ImageFont.load_default()

        # Badge
        draw.rectangle([25, 30, 390, 100], fill=badge_color)
        draw.text((207, 65), "",
                 font=fb, fill=(255,255,255), anchor='mm')

        # Título
        palabras = titulo_pantalla.upper().split()
        lineas, linea = [], ''
        for p in palabras:
            prueba = f"{linea} {p}".strip()
            if len(prueba) > 12:
                if linea: lineas.append(linea)
                linea = p
            else:
                linea = prueba
        if linea: lineas.append(linea)

        y_t = 130
        for lin in lineas[:2]:
            draw.text((ANCHO//2+3, y_t+3), lin, font=ft,
                     fill=(0,0,0), anchor='mm')
            draw.text((ANCHO//2, y_t), lin, font=ft,
                     fill=(255,255,255), anchor='mm')
            y_t += 80

        # Texto inferior
        draw.text((ANCHO//2+2, ALTO-60+2),
                 "VER VÍDEO COMPLETO EN EL CANAL 👆",
                 font=fs, fill=(0,0,0), anchor='mm')
        draw.text((ANCHO//2, ALTO-60),
                 "VER VÍDEO COMPLETO EN EL CANAL 👆",
                 font=fs, fill=(180,200,255), anchor='mm')

        return np.array(img)

    print('  🖼️  Procesando frames...')
    video_v = video.fl(lambda gf, t: procesar_frame(gf(t), t))

    ancho_f = 1080 if 1080 % 2 == 0 else 1078
    alto_f  = 1920 if 1920 % 2 == 0 else 1918

    ruta_temp        = WORK_DIR / f"short_v_{indice+1}_temp.mp4"
    ruta_audio_corto = WORK_DIR / f"short_v_{indice+1}_audio.aac"
    ruta_salida      = WORK_DIR / f"short_v_{indice+1}.mp4"

    print(f'  💾 Exportando vídeo base...')
    video_v.write_videofile(
        str(ruta_temp), fps=30, codec='libx264',
        audio=False, verbose=False, logger=None
    )

    try: video_v.close()
    except: pass
    try: video.close()
    except: pass
    gc.collect()

    # ── Extraer fragmento de audio ────────────────────────
    print('  ✂️  Extrayendo fragmento de audio...')
    cmd_audio = [
        'ffmpeg', '-y',
        '-i', str(audio_path),
        '-ss', str(inicio),
        '-t', str(duracion),
        '-c:a', 'aac',
        '-b:a', '192k',
        str(ruta_audio_corto)
    ]
    result_audio = subprocess.run(cmd_audio, capture_output=True, text=True)
    if result_audio.returncode != 0:
        print(f'  ⚠️  Error extrayendo audio: {result_audio.stderr[-100:]}')
    else:
        print(f'  ✅ Audio cortado correctamente')

    # ── Combinar vídeo + audio cortado ───────────────────
    print('  🎚️  Mezclando audio...')
    cmd = [
        'ffmpeg', '-y',
        '-i', str(ruta_temp),
        '-i', str(ruta_audio_corto),
        '-map', '0:v',
        '-map', '1:a',
        '-c:v', 'h264_nvenc',
        '-preset', 'p4', '-rc', 'vbr', '-cq', '23', '-b:v', '0',
        '-vf', f'scale={ancho_f}:{alto_f}',
        '-c:a', 'aac', '-b:a', '192k',
        '-shortest',
        str(ruta_salida)
    ]

    print('  🚀 Codificando con GPU (NVENC)...')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'  ⚠️  NVENC falló — usando libx264...')
        print(f'  Error: {result.stderr[-200:]}')
        cmd_fallback = [
            'ffmpeg', '-y',
            '-i', str(ruta_temp),
            '-i', str(ruta_audio_corto),
            '-map', '0:v',
            '-map', '1:a',
            '-c:v', 'libx264',
            '-vf', f'scale={ancho_f}:{alto_f}',
            '-c:a', 'aac', '-b:a', '192k',
            '-shortest',
            str(ruta_salida)
        ]
        subprocess.run(cmd_fallback, capture_output=True)
    else:
        print('  ✅ NVENC OK')

    # Limpiar temporales
    if ruta_temp.exists():
        ruta_temp.unlink()
    if ruta_audio_corto.exists():
        ruta_audio_corto.unlink()

    gc.collect()

    size_mb = ruta_salida.stat().st_size / 1024 / 1024
    print(f'  ✅ Short {indice+1} listo: {ruta_salida.name} ({size_mb:.0f} MB)')
    return ruta_salida


def subir_short_video(youtube, short_path: Path,
                      datos: dict, indice: int) -> str:
    import pytz
    from datetime import datetime, timedelta
    from googleapiclient.http import MediaFileUpload

    zona  = pytz.timezone('Europe/Madrid')
    ahora = datetime.now(zona)
    fecha = ahora.replace(hour=HORA_SHORTS, minute=0,
                          second=0, microsecond=0)
    if fecha <= ahora:
        fecha += timedelta(days=1)
    fecha += timedelta(days=indice * DIAS_ENTRE_SHORTS)

    titulo = (
        f"🌌 {datos['titulo'][:50]} "
        f"(Parte {indice+1}) #Shorts #Cosmos #UniversoProhibido"
    )[:100]

    body = {
        'snippet': {
            'title': titulo,
            'description': (
                f"Fragmento de: {datos['titulo']}\n\n"
                f"🚀 Vídeo completo en el canal 👆\n\n"
                f"#Shorts #Cosmos #Espacio #UniversoProhibido"
            ),
            'tags': ['shorts', 'cosmos', 'espacio',
                      'ciencia'],
            'categoryId': '28',
            'defaultLanguage': 'es'
        },
        'status': {
            'privacyStatus': 'private',
            'publishAt': fecha.isoformat(),
            'selfDeclaredMadeForKids': False
        }
    }

    media   = MediaFileUpload(str(short_path), mimetype='video/mp4',
                              resumable=True)
    request = youtube.videos().insert(
        part='snippet,status', body=body, media_body=media
    )
    response = None
    while response is None:
        status, response = request.next_chunk()
        if status:
            print(f'  📤 {int(status.progress()*100)}%')

    video_id = response['id']
    print(f'  ✅ https://youtube.com/shorts/{video_id}')
    print(f'  📅 Programado: {fecha.strftime("%d/%m/%Y a las %H:%M")}')
    return video_id


# ── EJECUTAR ──────────────────────────────────────────────
print('=' * 60)
print('📱 GENERANDO SHORTS DESDE VÍDEO LARGO')
print('=' * 60)

yt       = autenticar_youtube()
momentos = detectar_momentos(ruta_audio, N_SHORTS_VIDEO)
shorts_generados = []

for i, inicio in enumerate(momentos):
    print(f'\n🎬 Short {i+1}/{len(momentos)} — inicio: {inicio:.1f}s')
    try:
        ruta_short = crear_short_desde_video(
            ruta_video_final, ruta_audio,
            inicio, DURACION_SHORT_V, i, datos_video
        )
        shorts_generados.append(ruta_short)

        video_id = subir_short_video(yt, ruta_short, datos_video, i)

    except Exception as e:
        print(f'  ❌ Error Short {i+1}: {e}')
        import traceback
        traceback.print_exc()
        gc.collect()
        continue

print(f'\n{"="*60}')
print(f'✅ {len(shorts_generados)}/{N_SHORTS_VIDEO} Shorts generados')
for s in shorts_generados:
    print(f'  • {Path(s).name}')